In [53]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm
from datetime import datetime

import glob
from processing import *
from classical_estimates import classical_estimates
from fit_pv import *
from numpy.fft import fft2
from wavelengths import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
deadpix_file = '/home/ulyanov/data/solo/phi/dead_pixels/phi-fdt-deadpix_20250915T140003_V202609091415C_0569150100.fits'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [3]:
flat_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*cavity*.fits'))

print(flat_files)

['/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240330T050009_V202608262158C_0463300100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240926T114503_V202608262134C_0469260100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241016T113003_V202608262111C_0470160100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241027T233003_V202608262048C_0470270100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241202T123003_V202608262027C_0472020100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250119T210009_V202608262003C_0561190100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250310T080009_V202608261939C_0563100100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250915T140003_V202608261916C_0569150100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250923T000503_V202608261853C_0569230100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20260310T040003_V202608261828C_0663100

In [4]:
i = -2
cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

In [5]:
folder_blos = '/home/ulyanov/data/solo/phi/2026/blos_Q/'
folder_vlos = '/home/ulyanov/data/solo/phi/2026/vlos_/'

In [ ]:
temperature_constant = 4.01225e-2
tuning_constant = 3.513e-4
ref_wavelength = 6173.341
T0 = 61

In [72]:
import fnmatch
from connect import bob

sftp = bob()

top_dir = '/data/solo/phi/data/fmdb/l1/'
dirs = sorted(sftp.listdir(top_dir))

folder = '/home/ulyanov/data/solo/phi/2026/'

dates = []
angles = []
voltages = []
temperatures = []

for directory in dirs:
    if fnmatch.fnmatch(directory, '2026-0[4-7]*'):
        for file in sorted(sftp.listdir(top_dir + directory))[:1]:
            if fnmatch.fnmatch(file, '*fdt-alam*C_*.fits.gz'):
                print(file)

                remote_file = top_dir + directory + '/' + file
                local_file = 'temp.fits.gz'
                sftp.get(remote_file, local_file)

                data, header = process(local_file,
                                       dark_file=dark_file,
                                       deadpix_file=deadpix_file,
                                       prefilter_file=prefilter_file,
                                       #cavity_file=cavity_file,
                                       flatfield_file=flat_file,
                                       ghost_file=ghost_file,
                                       #distortion_file=distortion_file,
                                       #_realign=True,
                                       #_find_center=True,
                                       _demodulate=True,
                                       #_correct_fringes=True,
                                       #_correct_crosstalk=True,
                                       _calc_wavelengths=True,
                                       #_mask=True,
                                       )

                date = datetime.fromisoformat(header['DATE-OBS'])

                nx, ny = header['NAXIS2'], header['NAXIS1']
                x0, y0 = header['PXBEG2'] - 1, header['PXBEG1'] - 1

                contpos = header['CONTPOS'] - 1

                temp = np.zeros((2048,2048))
                temp[x0:x0+nx, y0:y0+ny] = data[contpos,2]

                fft = fft2(temp)
                angle = np.angle(fft[15,27])

                temperature = header['FGOV1PT1']
                wv = read_wavelengths(header)
                voltage = (wv[contpos] - ref_wavelength - temperature_constant * (temperature - T0)) / tuning_constant

                print(nx, ny, voltage, angle)

                dates += [date]
                voltages += [voltage]
                temperatures += [temperature]
                angles += [angle]


dates = np.array(dates)
voltages = np.array(voltages)
temperatures = np.array(temperatures)
angles = np.array(angles)

solo_L1_phi-fdt-alam_20260409T014503_V202604260832C_0644090501.fits.gz
1024 1024 -449.9928835761029 2.255371025098227
solo_L1_phi-fdt-alam_20260410T014503_V202604260832C_0644100501.fits.gz
1024 1024 -468.7233134087609 2.1007201894200156
solo_L1_phi-fdt-alam_20260411T020003_V202604270905C_0644110501.fits.gz
896 896 -491.60973526959157 2.024865824854748
solo_L1_phi-fdt-alam_20260412T020003_V202604270907C_0644120501.fits.gz
896 896 -515.1473811567689 1.8923211079878128
solo_L1_phi-fdt-alam_20260413T020003_V202604270908C_0644130501.fits.gz
896 896 -537.9234272709489 1.717042634576494
solo_L1_phi-fdt-alam_20260414T020003_V202604270909C_0644140501.fits.gz
896 896 -560.2120694584451 1.6265368767689994
solo_L1_phi-fdt-alam_20260415T003002_V202604230631C_0644150501.fits.gz
896 896 -578.9424992885141 1.471530276210264
solo_L1_phi-fdt-alam_20260416T020002_V202604230731C_0644160501.fits.gz
896 896 -605.9244947354855 1.3129724130995715
solo_L1_phi-fdt-alam_20260417T020002_V202605011531C_0644170501.

In [90]:
t = np.where(np.abs(temperatures - 61) < 1)[0]

voltages_ = voltages[t]
angles_ = angles[t] % (2 * np.pi)

t = np.where(np.all([voltages_ > 0,
                    voltages_ < 530], axis=0))[0]

k, b = np.polyfit(voltages_[t], angles_[t], 1)
k = 0.0085

plt.figure(figsize=(10,10))
plt.plot(voltages_, angles_, '.')
plt.plot(voltages_, (voltages_ * k + b) % (2 * np.pi) , '.')
plt.tight_layout()

In [86]:
k, b

(np.float64(0.007602338202327306), np.float64(1.9711751245759415))